# Classification

In [1]:
import numpy as np
import os

from pathlib import Path
from logistic_regression import evaluate_split, add_bias, train_threshold_models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [2]:
X_train=np.load("X_train.npy")
X_val=np.load("X_val.npy")
X_test=np.load("X_test.npy")

In [3]:
y_train_classified=np.load("y_train_classified.npy").ravel()
y_val_classified=np.load("y_val_classified.npy").ravel()
y_test_classified=np.load("y_test_classified.npy").ravel()

## KNN

In [8]:
for i in [9,11,13,15,17]:
    knn=KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train_classified)
    y_val_pred_classification=knn.predict(X_val)
    print('_'*10)
    print(i)
    print(classification_report(y_val_classified,y_val_pred_classification))

__________
9
              precision    recall  f1-score   support

           0       0.78      0.86      0.82     38124
           1       0.60      0.60      0.60     34004
           2       0.79      0.68      0.73     27872

    accuracy                           0.72    100000
   macro avg       0.72      0.71      0.72    100000
weighted avg       0.72      0.72      0.72    100000

__________
11
              precision    recall  f1-score   support

           0       0.79      0.87      0.83     38124
           1       0.62      0.62      0.62     34004
           2       0.81      0.69      0.75     27872

    accuracy                           0.74    100000
   macro avg       0.74      0.73      0.73    100000
weighted avg       0.74      0.74      0.74    100000

__________
13
              precision    recall  f1-score   support

           0       0.81      0.88      0.84     38124
           1       0.63      0.63      0.63     34004
           2       0.81      0.70 

In [9]:
knn_model=KNeighborsClassifier(n_neighbors=17)
knn_model.fit(X_train,y_train_classified)
y_val_pred_classification=knn_model.predict(X_val)
print(classification_report(y_val_classified,y_val_pred_classification))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85     38124
           1       0.65      0.66      0.65     34004
           2       0.83      0.72      0.77     27872

    accuracy                           0.76    100000
   macro avg       0.77      0.76      0.76    100000
weighted avg       0.76      0.76      0.76    100000



## Logistic Regression

In [4]:
java17_home = Path("/usr/lib/jvm/java-17-openjdk-amd64")
if java17_home.exists():
    os.environ["JAVA_HOME"] = str(java17_home)
    os.environ["PATH"] = f"{java17_home / 'bin'}:{os.environ.get('PATH', '')}"

try:
    from pyspark import StorageLevel
    from pyspark.sql import SparkSession
except ImportError as exc:
    raise ImportError(
        "Install pyspark first. If you use the project venv, run: .venv/bin/pip install pyspark"
    ) from exc

In [5]:
sample_size = None
rng = np.random.default_rng(42)

if sample_size is None:
    X_train_used = X_train
    y_train_used = y_train_classified
else:
    sample_size = min(sample_size, len(X_train))
    sample_idx = rng.choice(len(X_train), size=sample_size, replace=False)
    X_train_used = X_train[sample_idx]
    y_train_used = y_train_classified[sample_idx]

X_train_used = add_bias(X_train_used)
X_val_bias = add_bias(X_val)
X_test_bias = add_bias(X_test)

classes, counts = np.unique(y_train_used, return_counts=True)
print("Train sample:", X_train_used.shape, y_train_used.shape)
print("Validation:", X_val_bias.shape, y_val_classified.shape)
print("Test:", X_test_bias.shape, y_test_classified.shape)
print("Classes:", classes)
print("Train class counts:", dict(zip(classes.tolist(), counts.tolist())))

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("spark-map-reduce-ordinal-logistic-regression")
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "6g"))
    .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "6g"))
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

num_partitions = max(2, min(8, sc.defaultParallelism))
partition_edges = np.linspace(0, len(y_train_used), num_partitions + 1, dtype=int)
train_batches = [
    (
        np.ascontiguousarray(X_train_used[partition_edges[i]:partition_edges[i + 1]], dtype=np.float32),
        np.ascontiguousarray(y_train_used[partition_edges[i]:partition_edges[i + 1]], dtype=np.int64),
    )
    for i in range(num_partitions)
    if partition_edges[i] < partition_edges[i + 1]
]

train_rdd = sc.parallelize(train_batches, numSlices=len(train_batches)).persist(StorageLevel.MEMORY_AND_DISK)
num_rows = train_rdd.map(lambda batch: int(batch[1].shape[0])).sum()

print("RDD partitions:", train_rdd.getNumPartitions())
print("Rows inside RDD:", num_rows)

threshold_weights = train_threshold_models(
    sc=sc,
    train_rdd=train_rdd,
    num_features=X_train_used.shape[1],
    thresholds=(1, 2),
    epochs=25,
    lr=0.8,
    reg=0.0001,
)

print("Weights shape:", threshold_weights.shape)
evaluate_split("Validation", X_val_bias, y_val_classified, threshold_weights)
evaluate_split("Test", X_test_bias, y_test_classified, threshold_weights)

spark.stop()

Train sample: (800000, 60) (800000,)
Validation: (100000, 60) (100000,)
Test: (100000, 60) (100000,)
Classes: [0 1 2]
Train class counts: {0: 305245, 1: 271332, 2: 223423}


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 00:57:37 WARN Utils: Your hostname, mohamed-ashraf-LOQ-15IRX9, resolves to a loopback address: 127.0.1.1; using 192.168.1.102 instead (on interface wlp0s20f3)
26/04/12 00:57:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 00:57:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


RDD partitions: 8
Rows inside RDD: 800000

Training threshold y >= 1


  epoch 01 | loss = 0.6931


  epoch 02 | loss = 0.6063


...

  epoch 01 | loss = 0.6931
  epoch 02 | loss = 0.6408
  epoch 05 | loss = 0.3894


  epoch 10 | loss = 0.2894


  epoch 15 | loss = 0.2529


  epoch 20 | loss = 0.2322


  epoch 25 | loss = 0.2185
Weights shape: (2, 60)

Validation accuracy: 0.8625
              precision    recall  f1-score   support

           0     0.9156    0.8984    0.9069     38124
           1     0.7895    0.8120    0.8006     34004
           2     0.8828    0.8749    0.8789     27872

    accuracy                         0.8625    100000
   macro avg     0.8627    0.8618    0.8621    100000
weighted avg     0.8636    0.8625    0.8629    100000

Confusion matrix:
[[34250  3874     0]
 [ 3157 27611  3236]
 [    0  3486 24386]]

Test accuracy: 0.8612
              precision    recall  f1-score   support

           0     0.9148    0.8974    0.9060     38032
           1     0.7851    0.8138    0.7992     33933
           2     0.8858    0.8696    0.8776     28035

    accuracy                         0.8612    100000
   macro avg     0.8619    0.8602    0.8609    100000
weighted avg     0.8627    0.8612    0.8618    100000

Confusion matrix:
[[34130  3902     0]
 [ 3178 27613  